# ASEAN extensions — observed cost sensitivity and pure Top-100 diagnostic

Notebook này không chạy lại pipeline chính. Nó dùng các output ASEAN đã có để:

1. Chạy lại PTCST ablation với chi phí một chiều bằng half-spread BID/ASK quan sát được.
2. Kiểm tra khả năng chạy robustness với pure market-cap Top-100.

Pure market-cap Top-100 không thay thế mã thiếu dữ liệu. Vì vậy notebook sẽ báo rõ số ngày có đủ 100 mã; nếu bằng 0 thì không chạy mô hình pure để tránh đổi protocol âm thầm.

In [ ]:
# Cell 1 — mount Drive và clone code mới nhất
from google.colab import drive
# Clear a stale Drive mount, then remount explicitly.
try:
    drive.flush_and_unmount()
except Exception as e:
    print('No previous Drive mount to flush:', e)
drive.mount('/content/drive', force_remount=True)
from pathlib import Path
import subprocess, sys, shutil, zipfile, json, pandas as pd
REPO_URL = 'https://github.com/maiphuowng205/kltn.git'
REPO = Path('/content/kltn')
if REPO.exists(): shutil.rmtree(REPO)
subprocess.run(['git','clone','--depth','1',REPO_URL,str(REPO)],check=True)
print(subprocess.check_output(['git','-C',str(REPO),'rev-parse','HEAD'],text=True).strip())

In [ ]:
# Cell 2 — lấy main country runs từ Drive
DRIVE_ROOT = Path('/content/drive/MyDrive')
MAIN_ROOT = Path('/content/asean_v1_country_runs')
folders = [p for p in DRIVE_ROOT.rglob('asean_v1_country_runs') if p.is_dir()]
zips = [p for p in DRIVE_ROOT.rglob('*asean_v1_country_runs*.zip') if p.is_file()]
if MAIN_ROOT.exists(): shutil.rmtree(MAIN_ROOT)
if folders:
    shutil.copytree(folders[0], MAIN_ROOT)
elif zips:
    with zipfile.ZipFile(zips[0]) as z: z.extractall('/content')
else:
    raise FileNotFoundError('Upload asean_v1_country_runs folder/ZIP to MyDrive first.')
COUNTRIES = ['indonesia','malaysia','philippines','singapore','thailand']
SELECTED_COUNTRIES = COUNTRIES
print('Main package:', MAIN_ROOT)

In [ ]:
# Cell 3 — chạy cost sensitivity theo half-spread BID/ASK quan sát được
# Quote audit dùng end-of-day BID/ASK, không phải tick-level implementation shortfall.
def run(cmd):
    print('$', ' '.join(map(str,cmd)))
    return subprocess.run(cmd, cwd=REPO, check=True)
run([sys.executable, str(REPO/'scripts'/'run_asean_quote_cost_audit.py'), '--country-root', str(MAIN_ROOT), '--countries', *SELECTED_COUNTRIES])
costs = {}
for country in SELECTED_COUNTRIES:
    p = MAIN_ROOT/country/'runs'/'v3_quote_cost_audit'/'quote_cost_metrics.json'
    costs[country] = float(json.loads(p.read_text())['median_half_spread_bps'])
print('Observed one-way cost bps:', costs)
for country, cost_bps in costs.items():
    data_root = MAIN_ROOT/country/'data'/'lseg_v3'
    forecast = MAIN_ROOT/country/'runs'/'v3_ptcst_ca_mvo'
    out = MAIN_ROOT/country/'runs'/'v3_ptcst_cost_observed'
    if not (out/'portfolio_metrics_summary.parquet').exists():
        run([sys.executable, str(REPO/'scripts'/'run_v3_ptcst_ablations.py'), '--data-root', str(data_root), '--forecast-run', str(forecast), '--run-dir', str(out), '--cost-bps', str(cost_bps)])
print('Observed-cost sensitivity complete.')

In [ ]:
# Cell 4 — pure market-cap Top-100 coverage diagnostic
# Upload asean_v1_pure_top100 folder/ZIP nếu muốn kiểm tra package pure.
pure_folders = [p for p in DRIVE_ROOT.rglob('asean_v1_pure_top100') if p.is_dir()]
pure_zips = [p for p in DRIVE_ROOT.rglob('*asean_v1_pure_top100*.zip') if p.is_file()]
PURE_ROOT = Path('/content/asean_v1_pure_top100')
if pure_folders:
    if PURE_ROOT.exists(): shutil.rmtree(PURE_ROOT)
    shutil.copytree(pure_folders[0], PURE_ROOT)
elif pure_zips:
    with zipfile.ZipFile(pure_zips[0]) as z: z.extractall('/content')
else:
    PURE_ROOT = None
    print('Chưa upload pure Top-100 package; bỏ qua diagnostic pure.')
if PURE_ROOT is not None:
    report = json.loads((PURE_ROOT/'reports'/'pure_top100_report.json').read_text())
    print(json.dumps(report, indent=2))
    coverage = pd.read_csv(PURE_ROOT/'reports'/'pure_top100_coverage.csv')
    display(coverage)
    if int(report['outputs']['full_100_dates']) == 0:
        print('KẾT LUẬN: pure market-cap Top-100 không có ngày đủ 100 mã; không chạy model pure trong fixed-100 protocol.')
    else:
        print('Có ngày đủ 100 mã; cần một run ID riêng trước khi chạy model pure.')

In [ ]:
# Cell 5 — tổng hợp và lưu kết quả extension về Drive
OUT = Path('/content/asean_v1_extension_results')
if OUT.exists(): shutil.rmtree(OUT)
OUT.mkdir(parents=True)
rows=[]
for country in SELECTED_COUNTRIES:
    p=MAIN_ROOT/country/'runs'/'v3_ptcst_cost_observed'/'portfolio_metrics_summary.parquet'
    if p.exists():
        d=pd.read_parquet(p); d.insert(0,'country',country); rows.append(d)
if rows: pd.concat(rows,ignore_index=True).to_csv(OUT/'ptcst_observed_cost_sensitivity.csv',index=False)
if PURE_ROOT is not None and (PURE_ROOT/'reports'/'pure_top100_coverage.csv').exists(): shutil.copy2(PURE_ROOT/'reports'/'pure_top100_coverage.csv', OUT/'pure_top100_coverage.csv')
shutil.copy2(MAIN_ROOT/'asean_quote_cost_audit_summary.json', OUT/'asean_quote_cost_audit_summary.json')
DRIVE_OUT=Path('/content/drive/MyDrive/kltn/asean_v1_extension_results')
if DRIVE_OUT.exists(): shutil.rmtree(DRIVE_OUT)
shutil.copytree(OUT, DRIVE_OUT)
print('Đã lưu extension tại:', DRIVE_OUT)